# Tutorial: Lobo Fill Reports

**Audience:** Python users submitting orders through `lobo.Book`.

**Prerequisites:** A local development install (`make py`) and basic familiarity with limit and market orders.

**Learning goals:**
- Request no report, Summary only, Fills only, MarketImpact only, or all reports.
- Read each report object's rich text/HTML display.
- Choose `Execution.Simulation` without changing the book.
- Understand that report sections are calculated independently.


## Outline

1. Build a repeatable book fixture.
2. Compare no report, Summary, Fills, and MarketImpact.
3. Request all report sections together.
4. Compare simulation with the default mutating execution.
5. Try a short exercise and review common pitfalls.


In [1]:
from __future__ import annotations

from uuid import NAMESPACE_URL, uuid5

from lobo import Book, Execution, Reports
from lobo.orders import IcebergOrder, LimitOrder, MarketOrder, Side

SELLER = uuid5(NAMESPACE_URL, "lobo-reports-demo-seller")
BUYER = uuid5(NAMESPACE_URL, "lobo-reports-demo-buyer")


def make_book() -> Book:
    """Create 11 units of ask depth across prices 100, 101, and 102."""
    book = Book()
    book.fill(LimitOrder(100, 2, SELLER, Side.Sell))
    book.fill(LimitOrder(101, 3, SELLER, Side.Sell))
    book.fill(IcebergOrder(102, 2, SELLER, Side.Sell, 4))
    return book


def ask_state(book: Book) -> dict[str, int | None]:
    asks = book.orders.asks
    return {
        "best_ask": book.best_ask,
        "orders": asks.order_count,
        "visible_quantity": asks.visible_quantity,
        "hidden_quantity": asks.hidden_quantity,
    }


## 1. Inspect the fixture

Every example starts from a fresh copy of the same ask-side depth. This avoids hidden state and makes the report modes comparable.


In [2]:
fixture_book = make_book()
ask_state(fixture_book)


{'best_ask': 100, 'orders': 3, 'visible_quantity': 7, 'hidden_quantity': 4}

## 2. No report: the fastest path

Omitting `reports` uses `Reports()` internally. All flags default to `False`, so the fill result contains only `remaining_order_id` and `report` is `None`.


In [3]:
default_reports = Reports()
default_reports


include_fills,False
include_summary,False
include_market_impact,False


In [4]:
no_report_book = make_book()
no_report_result = no_report_book.fill(MarketOrder(8, BUYER, Side.Buy))
assert no_report_result.report is None
no_report_result


remaining_order_id,None
report,None


## 3. Summary only

Summary aggregates the request into filled quantity and realized integer price. It does not return the underlying fill records or calculate market impact.


In [5]:
summary_book = make_book()
summary_result = summary_book.fill(
    MarketOrder(8, BUYER, Side.Buy),
    Reports(include_summary=True),
)
assert summary_result.report.fills is None
assert summary_result.report.market_impact is None
summary_result.report.summary


order_id,bb631fd8-a536-4ffb-8016-b726693cf88c
trader_id,799a477d-d926-5f22-a21b-b8cca34ca2e5
side,Side.Buy
filled_quantity,8
realized_price,101


## 4. Fills only

Fills exposes each maker match as a `MatchedFill`. Summary remains `None`, so no additional aggregation is performed. The tape below makes price-level traversal easy to scan.


In [6]:
fills_book = make_book()

fills_result = fills_book.fill(
    MarketOrder(4, BUYER, Side.Buy),
    Reports(include_fills=True),
)
fills = fills_result.report.fills
assert fills_result.report.summary is None
assert fills_result.report.market_impact is None
fills


[MatchedFill(maker_order_id=UUID('6bce38c9-effc-4011-b18a-b770437579b8'), taker_order_id=UUID('417d8436-de69-4927-a3be-7c7b0c687507'), fill_quantity=2, maker_depleted=False, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=100, fill_time='2026-09-03T01:03:34.611279+00:00'),
 MatchedFill(maker_order_id=UUID('2f416259-acfb-48c9-bebb-1e7bee5fd064'), taker_order_id=UUID('417d8436-de69-4927-a3be-7c7b0c687507'), fill_quantity=2, maker_depleted=True, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=101, fill_time='2026-09-03T01:03:34.611282+00:00')]

In [7]:
[
    {
        "quantity": fill.fill_quantity,
        "price": fill.price,
        "maker_depleted": fill.maker_depleted,
    }
    for fill in fills
]


[{'quantity': 2, 'price': 100, 'maker_depleted': False},
 {'quantity': 2, 'price': 101, 'maker_depleted': True}]

## 5. MarketImpact only

MarketImpact describes execution quality against the pre-fill book: average and worst prices, slippage, levels consumed, and all available depth. It does not expose Fills or Summary unless requested.


In [8]:
impact_book = make_book()
impact_result = impact_book.fill(
    MarketOrder(8, BUYER, Side.Buy),
    Reports(include_market_impact=True),
)
assert impact_result.report.fills is None
assert impact_result.report.summary is None
impact_result.report.market_impact


avg_price,101.125
worst_price,102
slippage,2
slippage_bps,200.0
levels_consumed,3
total_quantity_available,11


## 6. Combine report sections

Flags are independent and can be combined. The `Report` display shows all requested sections together.


In [9]:
all_reports = Reports(
    include_fills=True,
    include_summary=True,
    include_market_impact=True,
)
combined_book = make_book()
combined_result = combined_book.fill(
    MarketOrder(4, BUYER, Side.Buy),
    all_reports,
)
combined_result.report


fills,"[MatchedFill(maker_order_id=UUID('ffc4ce35-a00e-4bb1-9ddf-814bc459c846'), taker_order_id=UUID('1b0d61cf-017a-4c6d-b0a2-876465bf77aa'), fill_quantity=2, maker_depleted=False, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=100, fill_time='2026-09-03T01:03:34.625238+00:00'), MatchedFill(maker_order_id=UUID('7df7e368-447d-48c6-aa97-8b7f7eb06dda'), taker_order_id=UUID('1b0d61cf-017a-4c6d-b0a2-876465bf77aa'), fill_quantity=2, maker_depleted=True, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=101, fill_time='2026-09-03T01:03:34.625240+00:00')]"
summary,"Summary(order_id=UUID('1b0d61cf-017a-4c6d-b0a2-876465bf77aa'), trader_id=UUID('799a477d-d926-5f22-a21b-b8cca34ca2e5'), side=Side.Buy, filled_quantity=4, realized_price=100)"
market_impact,"MarketImpact(avg_price=100.5, worst_price=101, slippage=1, slippage_bps=100.0, levels_consumed=2, total_quantity_available=11)"


## 7. Simulation versus mutation

`Execution.Simulation` calculates the requested reports but leaves resting orders unchanged. Omitting `execution` selects `Execution.Mutating`, which is the default.


In [10]:
simulation_book = make_book()
before = ask_state(simulation_book)
simulation_result = simulation_book.fill(
    MarketOrder(8, BUYER, Side.Buy),
    all_reports,
    execution=Execution.Simulation,
)
after_simulation = ask_state(simulation_book)
assert before == after_simulation

mutation_result = simulation_book.fill(
    MarketOrder(8, BUYER, Side.Buy),
    all_reports,
)
after_default_mutation = ask_state(simulation_book)
assert before != after_default_mutation

{
    "before": before,
    "after_simulation": after_simulation,
    "after_default_mutation": after_default_mutation,
    "simulated_filled_quantity": simulation_result.report.summary.filled_quantity,
    "mutated_filled_quantity": mutation_result.report.summary.filled_quantity,
}


{'before': {'best_ask': 100,
  'orders': 3,
  'visible_quantity': 7,
  'hidden_quantity': 4},
 'after_simulation': {'best_ask': 100,
  'orders': 3,
  'visible_quantity': 7,
  'hidden_quantity': 4},
 'after_default_mutation': {'best_ask': 102,
  'orders': 1,
  'visible_quantity': 1,
  'hidden_quantity': 2},
 'simulated_filled_quantity': 8,
 'mutated_filled_quantity': 8}

## Exercise

Request Fills and MarketImpact under simulation, but not Summary. Before running the next cell, predict which `Report` properties will be `None` and whether ask depth will change.


In [11]:
# Reference solution: edit the flags or quantity to continue experimenting.
exercise_book = make_book()
exercise_before = ask_state(exercise_book)
exercise_result = exercise_book.fill(
    MarketOrder(4, BUYER, Side.Buy),
    Reports(include_fills=True, include_market_impact=True),
    execution=Execution.Simulation,
)
assert exercise_result.report.summary is None
assert exercise_result.report.fills is not None
assert exercise_result.report.market_impact is not None
assert ask_state(exercise_book) == exercise_before
exercise_result.report


fills,"[MatchedFill(maker_order_id=UUID('0a1ac2eb-7abf-4b92-884c-c0bdc1ca2f6c'), taker_order_id=UUID('261f48cd-299e-4e86-bf31-2d8dd0cf41b0'), fill_quantity=2, maker_depleted=True, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=100, fill_time='2026-09-03T01:03:34.635629+00:00'), MatchedFill(maker_order_id=UUID('cff5a274-8fde-4401-901e-566696ad2f78'), taker_order_id=UUID('261f48cd-299e-4e86-bf31-2d8dd0cf41b0'), fill_quantity=2, maker_depleted=False, maker_trader_id=UUID('d20c385e-eec1-559e-833c-ab9f70d363b5'), price=101, fill_time='2026-09-03T01:03:34.635629+00:00')]"
summary,None
market_impact,"MarketImpact(avg_price=100.5, worst_price=101, slippage=1, slippage_bps=100.0, levels_consumed=2, total_quantity_available=11)"


In [12]:
make_book().orders.asks

visible_quantity,7
hidden_quantity,4
order_count,3


In [13]:
exercise_book.orders.asks

visible_quantity,7
hidden_quantity,4
order_count,3


## Pitfalls and next steps

- `Reports()` requests nothing; enable each section explicitly.
- Requesting Fills does not implicitly request Summary, and vice versa.
- Simulation returns hypothetical report data but does not rest an unmatched order or consume depth.
- Order UUIDs and fill timestamps are generated at runtime, so those identifiers differ between runs.

**Extension:** Try a sell market order against bid depth and observe that slippage remains side-aware.

You can now choose the minimum report set needed by an application and independently choose simulated or mutating execution.
